# Forecast Inference — uso prático

API limpa para prever preço de qualquer item, com intervalos de confiança.

**Modelos disponíveis:**
- `chronos` (default, Amazon, rápido, MAPE ~7-9%)
- `chronos-base` (Amazon, 4× lento, marginal improvement)
- `timesfm` (Google, ~igual ou melhor que Chronos)

**Returns**: `Forecast` com `.point`, `.lower`, `.upper`, `.dates`, `.as_dataframe()`

In [ ]:
import sys; sys.path.insert(0, "..")
from forecasting import forecast, list_available_models
from forecasting.inference import forecast_batch
import polars as pl
import matplotlib.pyplot as plt

print("Available models:", list_available_models())

## 1. Previsão para 1 item — Chronos h=7

In [ ]:
f = forecast("AK-47 | Redline (Field-Tested)", horizon=7, model="chronos")
print(f)
f.as_dataframe()

## 2. Plot — actual + forecast + band

In [ ]:
# Histórico recente + forecast
from forecasting.inference import _load_series
hist = _load_series("AK-47 | Redline (Field-Tested)").tail(60)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hist["date"], hist["price"], "k-", label="actual (60 last days)", lw=1.5)
ax.plot([f.context_last_date] + f.dates, [f.context_price, *f.point],
        "b--o", label="forecast", lw=1.5, markersize=4)
ax.fill_between(f.dates, f.lower, f.upper, alpha=0.2, color="blue", label="80% band")
ax.axvline(f.context_last_date, color="gray", ls=":")
ax.set_title(f"{f.name} — {f.model_used} h={f.horizon}")
ax.set_ylabel("USD")
ax.legend()
plt.show()

## 3. Horizonte maior (h=30)

In [ ]:
f30 = forecast("AWP | Asiimov (Field-Tested)", horizon=30, model="chronos")
print(f30)

hist = _load_series("AWP | Asiimov (Field-Tested)").tail(120)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hist["date"], hist["price"], "k-", label="actual", lw=1.5)
ax.plot(f30.dates, f30.point, "b-", label="point forecast", lw=2)
ax.fill_between(f30.dates, f30.lower, f30.upper, alpha=0.2, color="blue", label="80% band")
ax.axvline(f30.context_last_date, color="gray", ls=":")
ax.set_title(f"{f30.name} — h=30")
ax.legend()
plt.show()

## 4. Batch — vários items de uma vez

Útil para construir um dashboard de previsões diárias.

In [ ]:
items = [
    "AK-47 | Redline (Field-Tested)",
    "AWP | Asiimov (Field-Tested)",
    "M4A1-S | Cyrex (Minimal Wear)",
    "Glock-18 | Water Elemental (Field-Tested)",
    "USP-S | Kill Confirmed (Field-Tested)",
]
batch = forecast_batch(items, horizon=7, model="chronos")
print(batch)

## 5. Implied direction — vai subir ou descer?

In [ ]:
# Para cada item: comparar point_h7 com anchor_price
summary = (
    batch.group_by("name")
         .agg([
             pl.col("anchor_price").first().alias("now"),
             pl.col("point").last().alias("forecast_7d"),
             pl.col("lower").last().alias("lower_7d"),
             pl.col("upper").last().alias("upper_7d"),
         ])
         .with_columns(
             ((pl.col("forecast_7d") - pl.col("now")) / pl.col("now") * 100).round(1).alias("change_pct"),
             pl.when(pl.col("forecast_7d") > pl.col("now") * 1.02).then(pl.lit("UP"))
               .when(pl.col("forecast_7d") < pl.col("now") * 0.98).then(pl.lit("DOWN"))
               .otherwise(pl.lit("FLAT")).alias("direction"),
         )
         .sort("change_pct", descending=True)
)
print(summary)

## 6. Comparar modelos no mesmo item

Quando tiveres dúvida, corre os 2 modelos e compara o sinal.

In [ ]:
target = "AK-47 | Redline (Field-Tested)"
fc = forecast(target, horizon=7, model="chronos")
try:
    ft = forecast(target, horizon=7, model="timesfm")
    print(f"Chronos:  {fc.point[-1]:.2f} (h7)   band [{fc.lower[-1]:.2f}, {fc.upper[-1]:.2f}]")
    print(f"TimesFM:  {ft.point[-1]:.2f} (h7)   band [{ft.lower[-1]:.2f}, {ft.upper[-1]:.2f}]")
    print(f"Anchor:   {fc.context_price:.2f}")
except Exception as e:
    print(f"TimesFM falhou: {e}")
    print(f"Só Chronos: {fc.point[-1]:.2f}")

## Conclusões

- Para uso diário: **Chronos** chega.
- Para o intervalo de confiança: usa `f.lower` e `f.upper` (80%, q10-q90).
- Para tomar decisão de compra/venda: vê se `forecast_7d > anchor * 1.05` (esperar 5%+ subida).
- ⚠ Os intervals podem estar mal calibrados em items muito voláteis — sempre confirma visualmente no plot.